In [1]:
import os
from langchain_community.document_loaders import TextLoader, DirectoryLoader # To read the ppt files
from langchain_text_splitters import CharacterTextSplitter # For Chunking
from langchain_openai import OpenAIEmbeddings # Convert Chunks into vector embeddings
from langchain_chroma import Chroma #Chroma vector database to store the vector embeddings
from dotenv import load_dotenv

load_dotenv()

False

In [2]:
def load_documents(docs_path="docs"):
    """Load all text files from the docs directory"""
    print(f"Loading documents from {docs_path}...")
    
    # Check if docs directory exists
    if not os.path.exists(docs_path):
        raise FileNotFoundError(f"The directory {docs_path} does not exist. Please create it and add your company files.")
    
    # Load all .txt files from the docs directory
    loader = DirectoryLoader(
        path=docs_path,
        glob="*.txt", # Only look for txt files.
        loader_cls=TextLoader,
        loader_kwargs={"encoding": "utf-8"}
    )
    
    documents = loader.load()
    
    if len(documents) == 0:
        raise FileNotFoundError(f"No .txt files found in {docs_path}. Please add your company documents.")
    
   
    for i, doc in enumerate(documents[:2]):  # Show first 2 documents
        print(f"\nDocument {i+1}:")
        print(f"  Source: {doc.metadata['source']}")
        print(f"  Content length: {len(doc.page_content)} characters")
        print(f"  Content preview: {doc.page_content[:100]}...")
        print(f"  metadata: {doc.metadata}")

    return documents

def main():
    print("Main function")

    # Load the files
    documents = load_documents(docs_path="docs")
    # Chunking the files
    # Embedding and storing in Vector VB

if __name__ == "__main__":
    main()

Main function
Loading documents from docs...

Document 1:
  Source: docs\samsong_electronics.txt
  Content length: 13852 characters
  Content preview: Samsung Electronics Co., Ltd. (SEC; stylized as SΛMSUNG; Korean: 삼성전자; lit. 'Tristar Electronics') i...
  metadata: {'source': 'docs\\samsong_electronics.txt'}

Document 2:
  Source: docs\toyota.txt
  Content length: 34742 characters
  Content preview: Toyota Motor Corporation (Japanese: トヨタ自動車株式会社, Hepburn: Toyota Jidōsha kabushikigaisha; IPA: [toꜜjo...
  metadata: {'source': 'docs\\toyota.txt'}


In [3]:
def split_documents(documents, chunk_size=1000, chunk_overlap=0):
    """Split documents into smaller chunks with overlap"""
    print("Splitting documents into chunks...")
    
    text_splitter = CharacterTextSplitter(
        chunk_size=chunk_size, 
        chunk_overlap=chunk_overlap
    )
    
    chunks = text_splitter.split_documents(documents)
    
    if chunks:
    
        for i, chunk in enumerate(chunks[:5]):
            print(f"\n--- Chunk {i+1} ---")
            print(f"Source: {chunk.metadata['source']}")
            print(f"Length: {len(chunk.page_content)} characters")
            print(f"Content:")
            print(chunk.page_content)
            print("-" * 50)
        
        if len(chunks) > 5:
            print(f"\n... and {len(chunks) - 5} more chunks")
    
    return chunks

def main():
    print("Main function")

    # Load the files
    documents = load_documents(docs_path="docs")
    # Chunking the files
    chunks = split_documents(documents)
    # Embedding and storing in Vector VB

if __name__ == "__main__":
    main()

Created a chunk of size 1109, which is longer than the specified 1000
Created a chunk of size 1166, which is longer than the specified 1000
Created a chunk of size 1010, which is longer than the specified 1000
Created a chunk of size 1188, which is longer than the specified 1000
Created a chunk of size 1246, which is longer than the specified 1000
Created a chunk of size 1006, which is longer than the specified 1000


Main function
Loading documents from docs...

Document 1:
  Source: docs\samsong_electronics.txt
  Content length: 13852 characters
  Content preview: Samsung Electronics Co., Ltd. (SEC; stylized as SΛMSUNG; Korean: 삼성전자; lit. 'Tristar Electronics') i...
  metadata: {'source': 'docs\\samsong_electronics.txt'}

Document 2:
  Source: docs\toyota.txt
  Content length: 34742 characters
  Content preview: Toyota Motor Corporation (Japanese: トヨタ自動車株式会社, Hepburn: Toyota Jidōsha kabushikigaisha; IPA: [toꜜjo...
  metadata: {'source': 'docs\\toyota.txt'}
Splitting documents into chunks...

--- Chunk 1 ---
Source: docs\samsong_electronics.txt
Length: 498 characters
Content:
Samsung Electronics Co., Ltd. (SEC; stylized as SΛMSUNG; Korean: 삼성전자; lit. 'Tristar Electronics') is a South Korean multinational major appliance and consumer electronics corporation founded in 1969 and headquartered in Yeongtong District, Suwon, South Korea.[1] It is the pinnacle of the Samsung chaebol, accounting for 70% of

In [4]:
def create_vector_store(chunks, persist_directory="db/chroma_db"):
    """Create and persist ChromaDB vector store"""
    print("Creating embeddings and storing in ChromaDB...")
        
    embedding_model = OpenAIEmbeddings(model="text-embedding-3-small")
    
    # Create ChromaDB vector store
    print("--- Creating vector store ---")
    vectorstore = Chroma.from_documents(
        documents=chunks,
        embedding=embedding_model,
        persist_directory=persist_directory, 
        collection_metadata={"hnsw:space": "cosine"}
    )
    print("--- Finished creating vector store ---")
    
    print(f"Vector store created and saved to {persist_directory}")
    return vectorstore

def main():
    print("Main function")

    # Load the files
    documents = load_documents(docs_path="docs")
    # Chunking the files
    chunks = split_documents(documents)
    # Embedding and storing in Vector VB
    vector_store = create_vector_store(chunks)

if __name__ == "__main__":
    main()

Created a chunk of size 1109, which is longer than the specified 1000
Created a chunk of size 1166, which is longer than the specified 1000
Created a chunk of size 1010, which is longer than the specified 1000
Created a chunk of size 1188, which is longer than the specified 1000
Created a chunk of size 1246, which is longer than the specified 1000
Created a chunk of size 1006, which is longer than the specified 1000


Main function
Loading documents from docs...

Document 1:
  Source: docs\samsong_electronics.txt
  Content length: 13852 characters
  Content preview: Samsung Electronics Co., Ltd. (SEC; stylized as SΛMSUNG; Korean: 삼성전자; lit. 'Tristar Electronics') i...
  metadata: {'source': 'docs\\samsong_electronics.txt'}

Document 2:
  Source: docs\toyota.txt
  Content length: 34742 characters
  Content preview: Toyota Motor Corporation (Japanese: トヨタ自動車株式会社, Hepburn: Toyota Jidōsha kabushikigaisha; IPA: [toꜜjo...
  metadata: {'source': 'docs\\toyota.txt'}
Splitting documents into chunks...

--- Chunk 1 ---
Source: docs\samsong_electronics.txt
Length: 498 characters
Content:
Samsung Electronics Co., Ltd. (SEC; stylized as SΛMSUNG; Korean: 삼성전자; lit. 'Tristar Electronics') is a South Korean multinational major appliance and consumer electronics corporation founded in 1969 and headquartered in Yeongtong District, Suwon, South Korea.[1] It is the pinnacle of the Samsung chaebol, accounting for 70% of

RateLimitError: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}